[⬅ Back to the examples](https://anywidgetinstruments.github.io/anywidget-instruments-industrial/examples/)

# Filling line operator station

A bottle filling line with its operator station, built from the industrial
objects:

* the machine follows the **ISA-TR88.00.02** state model (`StateMachine`):
  Reset, Start, Hold, Suspend, Stop, Abort;
* a **HAND / OFF / AUTO** selector and a **stack light** at the machine;
* the product is heated to its filling temperature by a **PID loop**
  (`PID` + `PIDFaceplate`);
* the fill weight of each bottle is shown against its target on an
  **analog indicator** (grey unless out of limits);
* an **annunciator** (ISA-18.1 sequence R, first out) and an **alarm list**
  (ISA-18.2, with shelving) report the abnormal situations.

> **The code is hidden.** To see a cell's code, click the `⋯` bar above its result; to see all of it, choose **View ▸ Expand All Code**.

**References:** ISA-TR88.00.02 (PackML machine states), ISA-18.1
(annunciator sequences), ISA-18.2 / IEC 62682 (alarm list), ISA-101 (PID
faceplate, analog indicator), IEC 60073 (stack light colors). These
standards inspired the widgets; the library does not claim conformity with
them ([Standards and
references](https://anywidgetinstruments.github.io/anywidget-instruments-industrial/standards/)).

In [ ]:
import os
import threading
import time

import anywidget_instruments as ai

# How long the live simulation runs (seconds). The documentation build and the
# test-suite set AWI_EXAMPLE_SECONDS to run the examples briefly.
RUN_SECONDS = float(os.environ.get("AWI_EXAMPLE_SECONDS", "900"))

In [ ]:
ai.theme_switch()  # light / system / dark theme (STYLE-008)

In [ ]:
machine = ai.StateMachine(label="Line state (PackML)", size=(720, 380))
selector = ai.SelectorSwitch("AUTO", label="Line control", size=(140, 130))
light = ai.StackLight(
    tiers=["red", "amber", "green"],
    labels=["Fault", "Attention", "Producing"],
    label="Stack light",
    size=(150, 150),
)
heater = ai.PIDFaceplate(
    tag="TIC-101",
    label="Product temperature",
    unit="°C",
    pv_max=90,
    sp_min=20,
    sp_max=80,
    confirm_delta=10,
    hi=70,
    hihi=78,
    sp_tracking=True,
    modes=["MAN", "AUTO"],
    controller=ai.PID(kp=4.0, ti=40.0, sp=55.0),
)
weight = ai.AnalogIndicator(
    500,
    min=470,
    max=530,
    unit="g",
    format="%.1f",
    normal_lo=495,
    normal_hi=508,
    target=502,
    lo=490,
    hi=515,
    label="Fill weight",
    size=(300, 56),
)
count = ai.SevenSegment(0, digits=6, decimals=0, label="Bottles", size=(180, 60))
annunciator = ai.Annunciator(
    [
        ("TAH-101", "Product temp high", "red"),
        ("LAL-201", "Buffer tank low", "amber"),
        ("WA-301", "Fill weight off", "amber"),
        ("XA-401", "Capper jam", "amber"),
    ],
    sequence="R",
    first_out=True,
    columns=4,
    label="Annunciator",
    size=(460, 110),
)
alarms = ai.AlarmList(label="Alarm list", size=(640, 200))

ai.Panel(
    [
        ai.Panel([machine, ai.Panel([selector, light], columns=1)], columns=2),
        ai.Panel([heater, ai.Panel([weight, count, annunciator], columns=1)], columns=2),
        alarms,
    ],
    columns=1,
)

In [ ]:
LIGHT = {  # stack light pattern per state
    "Execute": {"green": "on"},
    "Starting": {"green": "blink"},
    "Completing": {"green": "blink"},
    "Held": {"amber": "on"},
    "Suspended": {"amber": "blink"},
    "Aborted": {"red": "blink"},
    "Aborting": {"red": "on"},
    "Stopped": {},
}


@machine.on_change
def _stack_light(change):
    pattern = LIGHT.get(change["new"], {"amber": "on"})
    light.value = [pattern.get(c, "off") for c in light.tiers]


@selector.on_change
def _selector(change):
    # OFF stops the line; HAND takes the heater loop in manual
    if change["new"] == "OFF" and "Stop" in machine.available_commands:
        machine.command("Stop")
    heater.loop_mode = "MAN" if change["new"] == "HAND" else "AUTO"


_active = {}


def alarm(tag, active, message, priority="high"):
    """Drive the annunciator window and the alarm list together, on changes only."""
    if _active.get(tag, False) == active:
        return
    _active[tag] = active
    annunciator.set(tag, active)
    if active:
        alarms.raise_alarm(tag, message, source=tag, priority=priority)
    else:
        alarms.clear_alarm(tag)


@alarms.on_event
def _log(event):
    print(f"{event['name']}: {event['alarm_id']}")


_stack_light({"new": machine.value})

In [ ]:
def simulate(duration=RUN_SECONDS, dt=0.25):
    """Line model: heated product, buffer tank, filler and capper."""
    temp, buffer, bottles, jam_at = 25.0, 60.0, 0, 45.0
    acting_since = time.time()
    last_state = machine.value
    t0 = time.time()
    while time.time() - t0 < duration:
        t = time.time() - t0
        state = machine.value
        if state != last_state:
            acting_since, last_state = time.time(), state
        # acting states complete after 2 s (IND-062); Execute runs until stopped
        if machine.is_acting and time.time() - acting_since > 2.0:
            machine.state_complete()
        producing = state == "Execute"
        # heater loop: first-order thermal model, heating power = OP %
        op = heater.step(temp, dt)
        temp += dt / 60.0 * (-(temp - 20.0) + 0.9 * op)
        # buffer tank drains while producing and is refilled slowly
        buffer = min(100.0, max(0.0, buffer + dt * (1.2 - (2.0 if producing else 0.0))))
        if producing:
            bottles += 1
            fill = 502.0 + 6.0 * ((bottles * 37) % 11 - 5) / 5.0  # repeatable scatter
            with ai.batch():
                count.value = bottles
                weight.value = fill
        # capper jam once, after 45 s of simulation: the line suspends itself
        if producing and t > jam_at:
            jam_at = float("inf")
            alarm("XA-401", True, "Capper jam", "medium")
            machine.command("Suspend")
        alarm("TAH-101", temp > heater.hi, "Product temperature high", "high")
        alarm("LAL-201", buffer < 15.0, "Buffer tank low", "medium")
        alarm("WA-301", weight.alarm_level != "normal", "Fill weight out of limits", "low")
        time.sleep(dt)


threading.Thread(target=simulate, daemon=True).start()

**Things to try**

* Press **Reset**, then **Start**: the acting states (dashed boxes) end by
  themselves after two seconds and the line reaches **Execute**; the stack
  light turns green and the bottle counter runs.
* Change the temperature setpoint on the faceplate: a change of more than
  10 °C asks for confirmation. Set the selector to **HAND** to take the loop
  in manual (the setpoint then tracks the temperature when you return to AUTO).
* After 45 s the capper jams: the line suspends, the annunciator window
  flashes (it is marked **1ST**) and the alarm appears in the list. Acknowledge
  it, then **Unsuspend** the line; clear the ringback with **RESET**.
* Shelve a nuisance alarm from the alarm list and watch the shelved counter.